In [1]:
!pip install transformers==4.57.6 chromadb sentence-transformers langgraph langchain-core gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 85.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 67.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 83.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 75.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 14.4 MB/s eta 0:00:00
   ━━━━

In [2]:
# Kaggle 노트북에서 실행
# 어떤 함수들이 정의돼 있는지 확인
import types
funcs = [name for name, obj in globals().items() 
         if isinstance(obj, types.FunctionType)]
print(sorted(funcs))

[]


In [ ]:
import os, cv2, json, gc, re, random, traceback
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from datetime import datetime, timedelta
from datetime import time as dt_time  # 이름 충돌 방지
from collections import defaultdict
from typing import TypedDict, Literal, Optional, Dict, Any
from PIL import Image
import torchvision.transforms as T
from torchvision.transforms import transforms
from torchvision.transforms.functional import InterpolationMode
from transformers import (
    AutoModel, AutoTokenizer, AutoModelForCausalLM,
    DPTConfig, DPTForSemanticSegmentation
)
from sentence_transformers import SentenceTransformer
from huggingface_hub import login
import chromadb
from langgraph.graph import StateGraph, END
from langchain_core.tools import tool
import gradio as gr
 
def _get_secret(name: str) -> str:
    """Kaggle Secrets(Add-ons > Secrets)에 등록된 값을 우선 사용하고,
    없으면 환경변수를 사용한다. 코드에 비밀값을 하드코딩하지 않기 위함."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name, "")


HF_TOKEN = _get_secret("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN이 설정되지 않았습니다. Kaggle 노트북의 Add-ons > Secrets에 "
        "HF_TOKEN을 등록하세요."
    )
login(token=HF_TOKEN)
 
RESIZED_IMAGE_DIR  = "/kaggle/input/datasets/jjjjjiiin/crystal-val-images"
MODEL_PATH         = "/kaggle/input/datasets/jjjjjiiin/best-dinov3-dpt-model1/best_DINOv3_DPT_model.pth"
PADDING_INFO_PATH  = "/kaggle/input/datasets/jjjjjiiin/padding-info1/padding_info.json"
INFERENCE_RESULTS  = "/kaggle/working/inference_results/inference_results.json"
MASK_DIR           = "/kaggle/working/inference_results"
VLM_RESULTS_PATH   = "/kaggle/working/vlm_results.json"
KG_GRAPH_PATH      = Path("/kaggle/input/datasets/jjjjjiiin/full-302-change-event-kg-graph-image1/full_302_change_event_kg_graph_image.json")
OUTPUT_DIR         = "/kaggle/working/inference_results"
LOG_DIR            = "/kaggle/working/agent_logs"
REFERENCE_IMAGE    = "/kaggle/input/datasets/jjjjjiiin/camera1-20250829-001848/Camera1_20250829_001848.png"
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
 
with open(PADDING_INFO_PATH, "r") as f:
    padding_info = json.load(f)
 
val_files = sorted([
    f for f in os.listdir(RESIZED_IMAGE_DIR)
    if f.lower().endswith(".png")
])
 
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
 
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

In [4]:
def safe_label_id(text):
    return str(text).strip().replace(" ", "_").replace("/", "_").replace(":", "_")

## DinoV3 정의 함수

In [5]:
class DINOv3DPT(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.backbone = AutoModel.from_pretrained("facebook/dinov3-vits16-pretrain-lvd1689m")
        cfg = self.backbone.config
        hidden_size = cfg.hidden_size
        self.n_register = cfg.num_register_tokens
 
        dpt_config = DPTConfig(
            hidden_size=hidden_size,
            num_hidden_layers=cfg.num_hidden_layers,
            num_attention_heads=cfg.num_attention_heads,
            intermediate_size=cfg.intermediate_size,
            hidden_act=cfg.hidden_act,
            num_labels=num_classes,
            id2label={0: "no_change", 1: "change", 2: "strong_change"},
            label2id={"no_change": 0, "change": 1, "strong_change": 2},
            image_size=1024,
            patch_size=cfg.patch_size,
        )
        dpt_model = DPTForSemanticSegmentation(dpt_config)
        self.neck = dpt_model.neck
        self.head = dpt_model.head
        for param in self.backbone.parameters():
            param.requires_grad = False
 
    def forward(self, x):
        B, C, H, W = x.shape
        h, w = H // 16, W // 16
        num_patches = h * w
        outputs = self.backbone(x, output_hidden_states=True)
        hidden_states = outputs.hidden_states
        n = len(hidden_states)
        selected = []
        for i in [n//4, n//2, n*3//4, n-1]:
            feat = hidden_states[i]
            feat = torch.cat([feat[:, :1, :], feat[:, 1 + self.n_register:, :]], dim=1)
            selected.append(feat)
        out = self.head(self.neck(selected, patch_height=h, patch_width=w))
        return nn.functional.interpolate(out, size=(H, W), mode="bilinear", align_corners=False)
 
 
dinov3_model = DINOv3DPT(num_classes=3).to(device)
state_dict = torch.load(MODEL_PATH, map_location=device, weights_only=True)
new_state_dict = {
    k.replace("module.", "").replace("backbone.layer.", "backbone.model.layer."): v
    for k, v in state_dict.items()
}
missing, unexpected = dinov3_model.load_state_dict(new_state_dict, strict=False)
dinov3_model.eval()
print(f"DINOv3 로드 완료 | missing: {len(missing)} | unexpected: {len(unexpected)}")


config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/86.4M [00:00<?, ?B/s]

DINOv3 로드 완료 | missing: 204 | unexpected: 204


## DinoV3 추론 함수

In [6]:
def extract_timestamp(filename):
    try:
        parts = filename.replace(".png", "").split("_")
        d = parts[1]
        return f"{d[:4]}-{d[4:6]}-{d[6:8]}"
    except Exception:
        return "unknown"
 
 
def infer_single_image_with_dinov3(image_path, save_dir="/kaggle/working/agent_outputs", change_threshold=0.35):
    os.makedirs(save_dir, exist_ok=True)
    img_file = os.path.basename(image_path)
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"이미지를 읽을 수 없습니다: {image_path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_tensor = transform(img).unsqueeze(0).to(device)
 
    with torch.no_grad():
        output = dinov3_model(img_tensor)
        probs = torch.softmax(output, dim=1).squeeze(0)
        pred_mask = probs.argmax(dim=0)
        pred_mask[(pred_mask != 2) & (probs[1] > change_threshold)] = 1
        pred_mask = pred_mask.cpu().numpy()
 
    pad = padding_info.get(img_file, {"top": 0, "bottom": 0, "left": 0, "right": 0})
    h, w = pred_mask.shape
    if pad["top"] > 0:    pred_mask[:pad["top"], :] = 255
    if pad["bottom"] > 0: pred_mask[h-pad["bottom"]:, :] = 255
    if pad["left"] > 0:   pred_mask[:, :pad["left"]] = 255
    if pad["right"] > 0:  pred_mask[:, w-pad["right"]:] = 255
 
    valid = (pred_mask != 255).sum()
    if valid == 0:
        raise ValueError(f"유효 픽셀 없음: {img_file}")

    valid = (pred_mask != 255).sum() # 패딩 제외 유효 픽셀 수 
 
    no_change    = round(float((pred_mask == 0).sum() / 
                               valid * 100), 2)
    change       = round(float((pred_mask == 1).sum() / 
                               valid * 100), 2)
    strong_change= round(float((pred_mask == 2).sum() / 
                               valid * 100), 2)
    total_change = round(change + strong_change, 2)
 
    vis = np.zeros((*pred_mask.shape, 3), dtype=np.uint8)
    vis[pred_mask == 1] = [255, 255, 0]
    vis[pred_mask == 2] = [0, 0, 255]
    mask_path = os.path.join(save_dir, f"mask_{img_file}")
    cv2.imwrite(mask_path, cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))
 
    return {
        "filename": img_file, "timestamp": extract_timestamp(img_file),
        "no_change": no_change, "change": change,
        "strong_change": strong_change, "total_change": total_change,
        "mask_path": mask_path
    }

In [7]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
results = {}
for img_file in val_files:
    img_path = os.path.join(RESIZED_IMAGE_DIR, img_file)
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_tensor = transform(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = dinov3_model(img_tensor)
        probs = torch.softmax(output, dim=1).squeeze(0)
        pred_mask = probs.argmax(dim=0)
        pred_mask[(pred_mask != 2) & 
                   (probs[1] > 0.35)] = 1
        pred_mask = pred_mask.cpu().numpy()
    
    pad = padding_info[img_file]
    h, w = pred_mask.shape
    if pad['top'] > 0:    pred_mask[:pad['top'], :] = 255
    if pad['bottom'] > 0: pred_mask[h-pad['bottom']:, :] = 255
    if pad['left'] > 0:   pred_mask[:, :pad['left']] = 255
    if pad['right'] > 0:  pred_mask[:, w-pad['right']:] = 255
    
    valid_pixels = (pred_mask != 255).sum()
    no_change    = (pred_mask == 0).sum() / valid_pixels * 100
    change       = (pred_mask == 1).sum() / valid_pixels * 100
    strong_change= (pred_mask == 2).sum() / valid_pixels * 100
    
    results[img_file] = {
        'no_change':     round(float(no_change), 2),
        'change':        round(float(change), 2),
        'strong_change': round(float(strong_change), 2)
    }
    print(f"{img_file}: No Change {no_change:.1f}% | Change {change:.1f}% | Strong Change {strong_change:.1f}%")
    
    vis_mask = np.zeros((*pred_mask.shape, 3), 
                        dtype=np.uint8)
    vis_mask[pred_mask == 1] = [255, 255, 0] # 노랑
    vis_mask[pred_mask == 2] = [0, 0, 255]   # 파랑
    cv2.imwrite(os.path.join(OUTPUT_DIR, 
                             f"mask_{img_file}"), 
                cv2.cvtColor(vis_mask, cv2.COLOR_RGB2BGR))

with open(os.path.join(OUTPUT_DIR, "inference_results.json"), 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n추론 완료! {len(results)}개 처리")

Camera1_20250907_125503.png: No Change 97.4% | Change 2.1% | Strong Change 0.6%
Camera1_20250908_125504.png: No Change 97.2% | Change 2.2% | Strong Change 0.6%
Camera1_20250910_144435.png: No Change 96.9% | Change 2.5% | Strong Change 0.6%
Camera1_20250911_114440.png: No Change 96.2% | Change 2.9% | Strong Change 0.9%
Camera1_20250911_144436.png: No Change 94.3% | Change 4.4% | Strong Change 1.3%
Camera1_20250912_144436.png: No Change 94.2% | Change 4.4% | Strong Change 1.4%
Camera1_20250913_174439.png: No Change 92.7% | Change 5.4% | Strong Change 1.9%
Camera1_20250914_024438.png: No Change 91.0% | Change 6.7% | Strong Change 2.3%
Camera1_20250914_054440.png: No Change 90.2% | Change 7.2% | Strong Change 2.6%
Camera1_20250915_144439.png: No Change 89.9% | Change 7.4% | Strong Change 2.7%
Camera1_20250915_234436.png: No Change 89.2% | Change 7.8% | Strong Change 3.1%
Camera1_20250916_084436.png: No Change 89.1% | Change 7.8% | Strong Change 3.1%
Camera1_20250917_144439.png: No Change 8

## RAG 

In [8]:
# =============================================================
# Cell 5: RAG (notebook 버전 - 한국어, 풍부한 텍스트)
# =============================================================
 
rag_docs = [
    {"text": "전체 변화 영역이 5% 미만이므로 기준 이미지와 비교했을 때 변화가 매우 작으며, 눈에 띄는 차이가 거의 없는 상태로 해석된다. 대부분의 영역이 기준 상태를 유지하고 있어 결정화 진행 정도는 매우 미미한 수준으로 볼 수 있다.",
     "category": "total_change", "level": "변화 거의 없음", "min": 0.0, "max": 5.0},
    {"text": "전체 변화 영역이 5~10% 범위에 해당하므로 낮은 수준의 변화가 나타난 상태이다. 대부분의 영역은 기준 이미지와 유사하지만, 일부 영역에서 작은 변화가 확인되며 결정화 진행이 초기 단계 또는 제한적인 범위에서 관찰되는 상태로 볼 수 있다.",
     "category": "total_change", "level": "낮은 변화", "min": 5.0, "max": 10.0},
    {"text": "전체 변화 영역이 10~15% 범위에 해당하므로 중간 수준의 변화가 나타난 상태이다. 기준 이미지와 비교했을 때 변화 영역이 비교적 뚜렷하게 확인되며, 결정화 진행이 일부 영역에서 명확하게 관찰되는 단계로 해석할 수 있다.",
     "category": "total_change", "level": "중간 변화", "min": 10.0, "max": 15.0},
    {"text": "전체 변화 영역이 15% 이상이므로 높은 수준의 변화가 나타난 상태이다. 기준 이미지와 비교했을 때 변화 영역이 넓게 분포하며, 결정화 진행이 상당히 뚜렷하고 광범위하게 나타난 상태로 해석할 수 있다.",
     "category": "total_change", "level": "높은 변화", "min": 15.0, "max": 100.0},
    {"text": "강한 변화 영역이 1% 미만이므로 두드러진 강한 변화는 거의 관찰되지 않는다. 전체 변화가 존재하더라도 대부분은 약한 변화에 가까우며, 강한 결정화 변화가 뚜렷하게 형성된 상태는 아닌 것으로 해석된다.",
     "category": "strong_change", "level": "강한 변화 거의 없음", "min": 0.0, "max": 1.0},
    {"text": "강한 변화 영역이 1~2% 범위에 해당하므로 제한적인 강한 변화가 일부 나타난 상태이다. 강한 변화는 넓게 퍼져 있다기보다 작은 점 또는 제한된 영역 형태로 관찰될 가능성이 높으며, 결정화 변화가 부분적으로 강화되기 시작한 상태로 볼 수 있다.",
     "category": "strong_change", "level": "부분적 강한 변화", "min": 1.0, "max": 2.0},
    {"text": "강한 변화 영역이 2~6% 범위에 해당하므로 비교적 뚜렷한 강한 변화가 관찰되는 상태이다. 일부 특정 영역에서 강한 변화가 군집 또는 덩어리 형태로 나타날 수 있으며, 결정화 진행이 약한 변화 단계를 넘어 더 뚜렷한 형태로 나타나는 것으로 해석할 수 있다.",
     "category": "strong_change", "level": "뚜렷한 강한 변화", "min": 2.0, "max": 6.0},
    {"text": "강한 변화 영역이 6% 이상이므로 광범위하고 두드러진 강한 변화가 나타난 상태이다. 강한 변화 영역이 명확한 결정 형태의 덩어리처럼 관찰될 가능성이 높으며, 결정화 진행이 강하게 나타난 상태로 해석할 수 있다.",
     "category": "strong_change", "level": "광범위한 강한 변화", "min": 6.0, "max": 100.0},
]
 
embedding_model = SentenceTransformer("BAAI/bge-m3")
chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("rag_docs")
except:
    pass
collection = chroma_client.get_or_create_collection("rag_docs")
collection.add(
    documents=[d["text"] for d in rag_docs],
    metadatas=[{"category": d["category"], "level": d["level"], "min": d["min"], "max": d["max"]} for d in rag_docs],
    ids=[f"doc_{i}" for i in range(len(rag_docs))],
    embeddings=[embedding_model.encode(d["text"], normalize_embeddings=True).tolist() for d in rag_docs]
)
print("RAG 컬렉션 초기화 완료")
 
 
def retrieve_ratio_doc(value, category, debug=False):
    # 해당 category 문서 조회
    results = collection.get(where={"category": category})
    
    for doc, meta in zip(results['documents'], results['metadatas']):
        min_v = float(meta.get('min', -1.0))
        max_v = float(meta.get('max', -1.0))
        if min_v <= value < max_v:
            best = {"text": doc, 
                    "level": meta.get('level', '알 수 없음'),
                    "category": category, 
                    "min": min_v, "max": max_v}
            if debug:
                print(f"[RAG] category={category}, value={value:.2f}% → {best['level']}")
            return best, [best]
 
    # fallback: 유사도 검색
    query_embedding = embedding_model.encode(f"{category} {value}%", normalize_embeddings=True).tolist()
    results = collection.query(query_embeddings=[query_embedding], n_results=4, where={"category": category})
    candidates = []
    for i, doc_text in enumerate(results["documents"][0]):
        meta = results["metadatas"][0][i]
        candidates.append({
            "text": doc_text, 
            "category": meta["category"], 
            "level": meta["level"],
            "min": float(meta["min"]), 
            "max": float(meta["max"]),
        })
    best = candidates[0] if candidates else {"text": f"{category} {value}% 기준 문서 없음", "level": "미정"}
    return best, candidates
 
 
def make_rag_context(img_file, no_change, change, 
                     strong_change, total_change, 
                     total_doc, strong_doc):
    return f"""분석 대상 이미지: {img_file}
[세그멘테이션 비율]
- No Change: {no_change:.2f}% / 
  Change: {change:.2f}% / 
  Strong Change: {strong_change:.2f}% / 
  Total Change: {total_change:.2f}%
[RAG 해석]
- 전체 변화 수준: {total_doc["level"]} — {total_doc["text"]}
- 강한 변화 수준: {strong_doc["level"]} — {strong_doc["text"]}

[마스크 색상] 검정=No Change / 노랑=Change / 파랑=Strong Change""".strip()
 

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

RAG 컬렉션 초기화 완료


## VLM 로드

In [35]:
# =============================================================
# Cell 6: VLM 로드 (notebook 버전)
# =============================================================
 
USE_VLM = True
 
if "vlm_model" not in dir() or vlm_model is None:
    try:
        gc.collect()
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        model_name = "OpenGVLab/InternVL2_5-4B"
        processor = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        vlm_model = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto", trust_remote_code=True
        ).eval()
        print("VLM 로드 완료")
    except Exception as e:
        print("VLM 로드 실패:", e)
        USE_VLM = False
        vlm_model = None
        processor = None
else:
    print("VLM 이미 로드됨, 스킵")
 
 
def build_transform(input_size):
    return T.Compose([
        T.Lambda(lambda img: img.convert("RGB")),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])
 
 
def dynamic_preprocess(image, min_num=1, max_num=2, image_size=448):
    w, h = image.size
    aspect_ratio = w / h
    target_ratios = sorted(set(
        (i, j) for n in range(min_num, max_num + 1)
        for i in range(1, n + 1) for j in range(1, n + 1)
        if min_num <= i * j <= max_num
    ), key=lambda x: x[0] * x[1])
    best_ratio = min(target_ratios, key=lambda r: abs(r[0] / r[1] - aspect_ratio))
    resized = image.resize((image_size * best_ratio[0], image_size * best_ratio[1]))
    tf = build_transform(image_size)
    tiles = [tf(resized.crop((j*image_size, i*image_size, (j+1)*image_size, (i+1)*image_size)))
             for i in range(best_ratio[1]) for j in range(best_ratio[0])]
    tiles.append(tf(image.resize((image_size, image_size))))
    return torch.stack(tiles)
 
 
def make_vlm_prompt(rag_context):
    return f"""<image>
[참고 정보]
{rag_context}

[작업]
세그멘테이션 마스크를 분석하여 한국어로 3~4문장의 관찰 보고서를 작성하라.

설명 규칙:
- 노란색은 Change 영역이다.
- 파란색은 Strong Change 영역이다.
- 1~3문장: 참고 정보의 변화율과 변화 수준을 활용하여 현재 상태를 설명한다.
- 마지막 문장: 변화 영역의 분포 형태를 설명한다.
  (산발적 분포 / 군집 분포 / 균일 분포)

중요:
- 반드시 한국어로만 작성한다.
- 영어를 사용하지 않는다.
- 온도, pH, 원인, 위치는 언급하지 않는다.
- 참고 정보에 없는 수치는 생성하지 않는다.
"""
 
 
def run_vlm(mask_image_path, rag_context):
    if not USE_VLM or vlm_model is None or processor is None or rag_context is None:
        return "VLM skipped"
    image = Image.open(mask_image_path).convert("RGB")
    pixel_values = dynamic_preprocess(image).to(torch.float16).to(vlm_model.device)
    question = "<image>\n" + make_vlm_prompt(rag_context).replace("<image>", "").strip()
    response = vlm_model.chat(processor, pixel_values, question, dict(max_new_tokens=256, do_sample=False))
    return response.strip()
 
 
def clean_vlm_report(response):
    if response in ("VLM skipped", "VLM failed") or not isinstance(response, str) or len(response) < 10:
        return "VLM 분석을 사용하지 않았으며, RAG 기준으로 판단하였다."
    for phrase in ["<image>", "[참고 정보]", "[작업]", "작성 방식:", "규칙:"]:
        response = response.replace(phrase, "")
    return response.strip()
 

VLM 이미 로드됨, 스킵


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.



✅ [Validation 성공]
파일명: Camera1_20250906_125505.png / total_change: 2.50%
✅ Neo4j 직접 저장 완료: ChangeEvent_Camera1_20250906_125505
[KG] records 갱신 완료: 66개
INFO:     36.38.169.176:0 - "POST /analyze?image_path=%2Fkaggle%2Finput%2Fdatasets%2Fjjjjjiiin%2Fcrystal2%2FCamera1_20250906_125505.png HTTP/1.1" 200 OK


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.



✅ [Validation 성공]
파일명: Camera1_20250930_144436.png / total_change: 12.02%
✅ Neo4j 직접 저장 완료: ChangeEvent_Camera1_20250930_144436
[KG] records 갱신 완료: 66개
INFO:     36.38.169.176:0 - "POST /analyze?image_path=%2Fkaggle%2Finput%2Fdatasets%2Fjjjjjiiin%2Fcrystal2%2FCamera1_20250930_144436.png HTTP/1.1" 200 OK


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.



✅ [Validation 성공]
파일명: Camera1_20251014_135859.png / total_change: 15.56%
✅ Neo4j 직접 저장 완료: ChangeEvent_Camera1_20251014_135859
[KG] records 갱신 완료: 66개
INFO:     36.38.169.176:0 - "POST /analyze?image_path=%2Fkaggle%2Finput%2Fdatasets%2Fjjjjjiiin%2Fcrystal2%2FCamera1_20251014_135859.png HTTP/1.1" 200 OK


In [10]:
def record_to_text(r):
    if not r: return "해당 record 없음"
    return (
        f"이미지: {r.get('target_image', r.get('filename', 'unknown'))}\n"
        f"날짜: {r.get('date', r.get('timestamp', 'unknown'))}\n"
        f"total_change: {r.get('total_change', 'N/A')}%\n"
        f"change: {r.get('change', 'N/A')}% / strong_change: {r.get('strong_change', 'N/A')}%\n"
        f"change_level: {r.get('change_level', 'N/A')}\n"
        f"strong_change_level: {r.get('strong_change_level', 'N/A')}\n"
        f"daily_change_speed_per_day: {r.get('daily_change_speed_per_day', 'N/A')}%p/day\n"
        f"previous_date: {r.get('previous_date', 'N/A')}"
    )
 
 
def filter_records_by_date_range(records, start, end):
    result = []
    for r in records:
        dt = parse_record_datetime(r)
        if dt and start <= dt <= end:
            result.append(r)
    return result
 
 
def extract_korean_date_range(question, records):
    patterns = [
        r'(\d{4})년\s*(\d{1,2})월\s*(\d{1,2})일.*?~.*?(\d{4})년\s*(\d{1,2})월\s*(\d{1,2})일',
        r'(\d{4})-(\d{1,2})-(\d{1,2}).*?~.*?(\d{4})-(\d{1,2})-(\d{1,2})',
        r'(\d{4})년\s*(\d{1,2})월\s*(\d{1,2})일\s*부터\s*(\d{4})년\s*(\d{1,2})월\s*(\d{1,2})일',
    ]
    for p in patterns:
        m = re.search(p, question)
        if m:
            g = m.groups()
            try:
                start = datetime(int(g[0]), int(g[1]), int(g[2]))
                end = datetime(int(g[3]), int(g[4]), int(g[5]), 23, 59, 59)
                return start, end
            except:
                pass
    return None
 
 
def extract_single_date(question, records):
    patterns = [
        r'(\d{4})년\s*(\d{1,2})월\s*(\d{1,2})일',
        r'(\d{4})-(\d{1,2})-(\d{1,2})',
        r'(\d{4})\.(\d{1,2})\.(\d{1,2})',
        r'(\d{1,2})월\s*(\d{1,2})일',  # 연도 없는 패턴 추가
    ]

    # records에서 연도 자동 추출
    current_year = 2025
    if records:
        for r in records:
            date_str = r.get("date") or r.get("timestamp", "")
            if date_str:
                try:
                    current_year = int(date_str[:4])
                    break
                except:
                    pass

    for p in patterns:
        m = re.search(p, question)
        if m:
            g = m.groups()
            try:
                if len(g) == 2:  # 연도 없는 경우
                    dt = datetime(current_year, int(g[0]), int(g[1]))
                else:
                    dt = datetime(int(g[0]), int(g[1]), int(g[2]))
                return dt, dt
            except:
                pass
    return None
 
 
def extract_two_dates_for_comparison(question, records):
    patterns = [
        r'(\d{4})-(\d{1,2})-(\d{1,2})',
        r'(\d{4})년\s*(\d{1,2})월\s*(\d{1,2})일',
    ]
    dates_found = []
    for p in patterns:
        for m in re.finditer(p, question):
            g = m.groups()
            try:
                dates_found.append(datetime(int(g[0]), int(g[1]), int(g[2])))
            except:
                pass
        if len(dates_found) >= 2:
            return dates_found[0], dates_found[1]
    return None
 
 
def get_date_avg_table(records):
    date_map = defaultdict(list)
    for r in records:
        d = r.get("date") or r.get("timestamp")
        if d:
            date_map[d].append(safe_float(r.get("total_change")))
    result = []
    for d, vals in sorted(date_map.items()):
        avg = round(sum(vals) / len(vals), 4)
        try:
            dt = parse_date(d)
        except:
            continue
        result.append({"date": d, "date_dt": dt, "avg_total_change": avg, "image_count": len(vals)})
    return sorted(result, key=lambda x: x["date_dt"])
 
 
def get_avg_total_change_for_date(records, target_date):
    items = get_date_avg_table(records)
    for item in items:
        if item["date_dt"].date() == target_date.date():
            return item
    return None
 
 
def get_top_date_by_avg_total_change(records):
    items = get_date_avg_table(records)
    return max(items, key=lambda x: x["avg_total_change"]) if items else None
 
 
def get_low_date_by_avg_total_change(records):
    items = get_date_avg_table(records)
    return min(items, key=lambda x: x["avg_total_change"]) if items else None
 
 
def compare_two_dates_avg_total_change(records, date1, date2):
    if date1 > date2: date1, date2 = date2, date1
    d1i = get_avg_total_change_for_date(records, date1)
    d2i = get_avg_total_change_for_date(records, date2)
    days = abs((date2 - date1).days)
    if d1i is None or d2i is None:
        return {"date1": date1.strftime("%Y-%m-%d"), "date2": date2.strftime("%Y-%m-%d"),
                "date1_info": d1i, "date2_info": d2i, "diff": None, "abs_diff": None,
                "direction": "계산 불가", "days_between": days, "interval_change_speed_per_day": None}
    diff = round(d2i["avg_total_change"] - d1i["avg_total_change"], 4)
    return {"date1": d1i["date"], "date2": d2i["date"], "date1_info": d1i, "date2_info": d2i,
            "diff": diff, "abs_diff": round(abs(diff), 4),
            "direction": "증가" if diff > 0 else ("감소" if diff < 0 else "동일"),
            "days_between": days, "interval_change_speed_per_day": round(diff/days, 4) if days > 0 else None}
 
 
def get_baseline_speed_table(records):
    t = get_date_avg_table(records)
    if not t: return []
    base = t[0]
    result = []
    for item in t:
        dfb = (item["date_dt"] - base["date_dt"]).days
        delta = round(item["avg_total_change"] - base["avg_total_change"], 4)
        speed = 0.0 if dfb == 0 else round(delta / dfb, 4)
        result.append({**item, "baseline_date": base["date"], "baseline_avg_total_change": base["avg_total_change"],
                       "days_from_baseline": dfb, "baseline_change_delta": delta,
                       "baseline_change_speed_per_day": speed,
                       "direction": "증가" if delta > 0 else ("감소" if delta < 0 else "동일")})
    return result
 
 
def get_top_baseline_speed_date(records):
    c = [x for x in get_baseline_speed_table(records) if x["days_from_baseline"] > 0]
    return max(c, key=lambda x: x["baseline_change_speed_per_day"]) if c else None
 
 
def get_low_baseline_speed_date(records):
    c = [x for x in get_baseline_speed_table(records) if x["days_from_baseline"] > 0]
    return min(c, key=lambda x: x["baseline_change_speed_per_day"]) if c else None
 
 
def calculate_trend_change_speed(records):
    t = get_date_avg_table(records)
    if len(t) < 2: return None
    base = t[0]["date_dt"]
    xs = [(i["date_dt"] - base).days for i in t]
    ys = [i["avg_total_change"] for i in t]
    xm, ym = sum(xs)/len(xs), sum(ys)/len(ys)
    denom = sum((x-xm)**2 for x in xs)
    slope = round(sum((x-xm)*(y-ym) for x, y in zip(xs, ys))/denom, 4) if denom else 0.0
    return {"start_date": t[0]["date"], "end_date": t[-1]["date"], "date_count": len(t),
            "trend_change_speed_per_day": slope,
            "trend_direction": "증가 추세" if slope > 0 else ("감소 추세" if slope < 0 else "변화 거의 없음"),
            "date_avg_values": [{"date": i["date"], "avg_total_change": i["avg_total_change"], "image_count": i["image_count"]} for i in t]}
 
 
def get_summary(records):
    total = len(records)
    if total == 0:
        return {"total": 0, "avg_change": 0, "avg_strong": 0, "avg_total": 0, "avg_previous_speed": None,
                "change_level_count": {}, "strong_level_count": {}, "speed_level_count": {},
                "max_change_record": None, "min_change_record": None, "max_previous_speed_record": None,
                "slowest_previous_speed_record": None, "largest_decrease_previous_record": None,
                "top_baseline_speed_date": None, "trend": None}
    avg_change = sum(safe_float(r.get("change")) for r in records) / total
    avg_strong = sum(safe_float(r.get("strong_change")) for r in records) / total
    avg_total  = sum(safe_float(r.get("total_change")) for r in records) / total
    valid = [r for r in records if r.get("daily_change_speed_per_day") is not None]
    cl, sl, spl = {}, {}, {}
    for r in records:
        for d, k in [(cl, "change_level"), (sl, "strong_change_level"), (spl, "change_speed_level")]:
            v = r.get(k, "Unknown"); d[v] = d.get(v, 0) + 1
    return {
        "total": total, "avg_change": avg_change, "avg_strong": avg_strong, "avg_total": avg_total,
        "avg_previous_speed": sum(safe_float(r.get("daily_change_speed_per_day")) for r in valid)/len(valid) if valid else None,
        "change_level_count": cl, "strong_level_count": sl, "speed_level_count": spl,
        "max_change_record": max(records, key=lambda r: safe_float(r.get("total_change"))),
        "min_change_record": min(records, key=lambda r: safe_float(r.get("total_change"))),
        "max_previous_speed_record": max(valid, key=lambda r: safe_float(r.get("daily_change_speed_per_day"))) if valid else None,
        "slowest_previous_speed_record": min(valid, key=lambda r: abs(safe_float(r.get("daily_change_speed_per_day")))) if valid else None,
        "largest_decrease_previous_record": min(valid, key=lambda r: safe_float(r.get("daily_change_speed_per_day"))) if valid else None,
        "top_baseline_speed_date": get_top_baseline_speed_date(records),
        "trend": calculate_trend_change_speed(records)
    }

def neo4j_retrieve_records(question, top_k=5):
    q = question.lower()

    with get_driver().session() as session:

        # 단일 날짜 검색
        single_date = extract_single_date(question, [])
        if single_date:
            start, end = single_date
            result = session.run("""
                MATCH (n:Record)
                WHERE n.date = $date
                RETURN n
                ORDER BY n.total_change DESC
                LIMIT $limit
            """, date=start.strftime("%Y-%m-%d"), limit=top_k)
            records = [dict(r["n"]) for r in result]
            return {"type": "date_search", "start": start, "end": end, "records": records}

        # 날짜 범위 검색
        date_range = extract_korean_date_range(question, [])
        if date_range:
            start, end = date_range
            result = session.run("""
                MATCH (n:Record)
                WHERE n.date >= $start AND n.date <= $end
                RETURN n
                ORDER BY n.total_change DESC
                LIMIT $limit
            """, start=start.strftime("%Y-%m-%d"), end=end.strftime("%Y-%m-%d"), limit=top_k)
            records = [dict(r["n"]) for r in result]
            return {"type": "date_range_search", "start": start, "end": end, "records": records}

        # 변화율 최대
        if any(w in q for w in ["가장 큰", "최대", "높은", "많이"]):
            result = session.run("""
                MATCH (n:Record)
                RETURN n
                ORDER BY n.total_change DESC
                LIMIT $limit
            """, limit=top_k)
            records = [dict(r["n"]) for r in result]
            return {"type": "top_change", "records": records}

        # 변화율 최소
        if any(w in q for w in ["가장 작은", "최소", "낮은"]):
            result = session.run("""
                MATCH (n:Record)
                RETURN n
                ORDER BY n.total_change ASC
                LIMIT $limit
            """, limit=top_k)
            records = [dict(r["n"]) for r in result]
            return {"type": "low_change", "records": records}

        # 변화 속도
        if any(w in q for w in ["변화 속도", "속도"]):
            order = "ASC" if any(w in q for w in ["느린", "낮은", "최소"]) else "DESC"
            result = session.run(f"""
                MATCH (n:Record)
                WHERE n.daily_change_speed_per_day IS NOT NULL
                RETURN n
                ORDER BY n.daily_change_speed_per_day {order}
                LIMIT $limit
            """, limit=top_k)
            records = [dict(r["n"]) for r in result]
            return {"type": "top_previous_change_speed", "records": records}

        # 그래프 탐색 (특정 날짜의 연결 관계)
        if any(w in q for w in ["연결", "관계", "그래프"]):
            result = session.run("""
                MATCH (e:ChangeEvent)-[:OCCURRED_ON]->(d:Date)
                MATCH (e)-[:HAS_CHANGE_LEVEL]->(cl:ChangeLevel)
                RETURN e.change_event_id, d.date, cl.label, e.total_change
                ORDER BY e.total_change DESC
                LIMIT $limit
            """, limit=top_k)
            records = [dict(r) for r in result]
            return {"type": "graph_search", "records": records}

        # 전체 요약
        result = session.run("""
            MATCH (n:Record)
            RETURN n
            ORDER BY n.date ASC
        """)
        all_records = [dict(r["n"]) for r in result]
        return {"type": "summary", "summary": get_summary(all_records)}
 
def retrieve_records(question, records, top_k=5):
    q = question.lower()
 
    if any(w in q for w in ["전에", "이전에", "까지", "전의"]):
        single = extract_single_date(question, records)
        if single:
            end_date = datetime.combine(single[0].date(), time(23, 59, 59)) - timedelta(days=1)
            end_date = datetime.combine(end_date.date(), time(23, 59, 59))
            dts = [parse_record_datetime(r) for r in records if parse_record_datetime(r)]
            start_date = min(dts) if dts else None
            if start_date:
                filtered = filter_records_by_date_range(records, start_date, end_date)
                if any(w in q for w in ["가장 큰", "제일 큰", "높은", "많이", "최대", "변화가 큰", "큰 변화"]):
                    return {"type": "date_range_top_change", "start": start_date, "end": end_date,
                            "records": sorted(filtered, key=lambda r: safe_float(r.get("total_change")), reverse=True)[:top_k]}
                if any(w in q for w in ["가장 작은", "낮은", "최소", "변화가 작은", "작은 변화"]):
                    return {"type": "date_range_low_change", "start": start_date, "end": end_date,
                            "records": sorted(filtered, key=lambda r: safe_float(r.get("total_change")))[:top_k]}
                return {"type": "date_range_search", "start": start_date, "end": end_date, "records": filtered[:top_k]}
 
    if any(w in q for w in ["차이", "비교", "얼마나", "간의", "사이"]):
        two = extract_two_dates_for_comparison(question, records)
        if two:
            comp = compare_two_dates_avg_total_change(records, *two)
            t = "two_date_interval_speed" if any(w in q for w in ["변화 속도", "속도"]) else "two_date_total_change_diff"
            return {"type": t, "comparison": comp}
 
    if any(w in q for w in ["변화 속도", "속도", "speed"]):
        if any(w in q for w in ["기준", "baseline", "누적"]):
            return {"type": "low_baseline_change_speed" if any(w in q for w in ["가장 낮", "가장 느린", "최소"]) else "top_baseline_change_speed",
                    "best_date": get_low_baseline_speed_date(records) if "낮" in q or "느린" in q else get_top_baseline_speed_date(records)}
        if any(w in q for w in ["추세", "trend", "전체적으로"]):
            return {"type": "trend_change_speed", "trend": calculate_trend_change_speed(records)}
        valid = [r for r in records if r.get("daily_change_speed_per_day") is not None]
        if any(w in q for w in ["가장 느린", "slowest"]):
            return {"type": "slowest_previous_change_speed", "records": sorted(valid, key=lambda r: abs(safe_float(r.get("daily_change_speed_per_day"))))[:top_k]}
        if any(w in q for w in ["감소", "decrease"]):
            return {"type": "largest_decrease_previous_speed", "records": sorted(valid, key=lambda r: safe_float(r.get("daily_change_speed_per_day")))[:top_k]}
        return {"type": "top_previous_change_speed", "records": sorted(valid, key=lambda r: safe_float(r.get("daily_change_speed_per_day")), reverse=True)[:top_k]}
 
    date_range = extract_korean_date_range(question, records)
    if date_range:
        start, end = date_range
        filtered = filter_records_by_date_range(records, start, end)
        if any(w in q for w in ["가장 큰", "높은", "많이", "max"]):
            return {"type": "date_range_top_change", "start": start, "end": end,
                    "records": sorted(filtered, key=lambda r: safe_float(r.get("total_change")), reverse=True)[:top_k]}
        if any(w in q for w in ["가장 작은", "낮은", "min"]):
            return {"type": "date_range_low_change", "start": start, "end": end,
                    "records": sorted(filtered, key=lambda r: safe_float(r.get("total_change")))[:top_k]}
        return {"type": "date_range_search", "start": start, "end": end, "records": filtered[:top_k]}
 
    if any(w in q for w in ["어느 날짜", "날짜별"]):
        if any(w in q for w in ["가장 큰", "가장 높은", "최대"]):
            return {"type": "top_date_avg_change", "best_date": get_top_date_by_avg_total_change(records)}
        if any(w in q for w in ["가장 작은", "가장 낮은", "최소"]):
            return {"type": "low_date_avg_change", "best_date": get_low_date_by_avg_total_change(records)}
 
    single_date = extract_single_date(question, records)
    if single_date:
        start, end = single_date
        end = datetime.combine(end.date(), dt_time(23, 59, 59))
        filtered = filter_records_by_date_range(records, start, end)
        if any(w in q for w in ["가장 큰", "높은", "max"]): filtered = sorted(filtered, key=lambda r: safe_float(r.get("total_change")), reverse=True)
        elif any(w in q for w in ["가장 작은", "낮은", "min"]): filtered = sorted(filtered, key=lambda r: safe_float(r.get("total_change")))
        return {"type": "date_search", "start": start, "end": end, "records": filtered[:top_k]}
 
    m = re.search(r"(\d+(?:\.\d+)?)\s*%", q)
    if m and any(w in q for w in ["변화율", "total_change", "전체 변화"]):
        target = float(m.group(1))
        tolerance = 0.25
        exact = sorted([r for r in records if abs(safe_float(r.get("total_change")) - target) <= tolerance],
                       key=lambda r: abs(safe_float(r.get("total_change")) - target))
        if exact: return {"type": "total_change_value_search", "target_total": target, "tolerance": tolerance, "records": exact[:top_k], "matched": True}
        return {"type": "total_change_value_search", "target_total": target, "tolerance": tolerance,
                "records": sorted(records, key=lambda r: abs(safe_float(r.get("total_change")) - target))[:top_k], "matched": False}
 
    if any(w in q for w in ["가장 큰", "제일 큰", "높은", "많이", "max"]):
        return {"type": "top_change", "records": sorted(records, key=lambda r: safe_float(r.get("total_change")), reverse=True)[:top_k]}
    if any(w in q for w in ["가장 작은", "제일 작은", "낮은", "min"]):
        return {"type": "low_change", "records": sorted(records, key=lambda r: safe_float(r.get("total_change")))[:top_k]}
 
    return {"type": "summary", "summary": get_summary(records)}
 
 
def build_context(retrieved):
    rtype = retrieved["type"]
    if rtype == "summary":
        s = retrieved["summary"]
        if s["total"] == 0: return "KG에 record가 없습니다."
        return (f"[KG 전체 요약] 총 {s['total']}개\n"
                f"평균 total_change: {s['avg_total']:.4f}%\n"
                f"최대: {record_to_text(s['max_change_record'])}\n"
                f"최소: {record_to_text(s['min_change_record'])}\n"
                f"추세: {s['trend']['trend_change_speed_per_day'] if s['trend'] else '계산 불가'}%p/day")
    if rtype in ["two_date_total_change_diff", "two_date_interval_speed"]:
        comp = retrieved["comparison"]
        if comp["diff"] is None: return f"[두 날짜 비교] {comp['date1']}과 {comp['date2']} 중 KG record 없어 계산 불가."
        d1, d2 = comp["date1_info"], comp["date2_info"]
        return (f"[두 날짜 비교]\n{comp['date1']}: {d1['avg_total_change']}% / {comp['date2']}: {d2['avg_total_change']}%\n"
                f"차이: {comp['diff']}%p / 속도: {comp['interval_change_speed_per_day']}%p/day / 방향: {comp['direction']}")
    if rtype in ["top_baseline_change_speed", "low_baseline_change_speed"]:
        bd = retrieved["best_date"]
        if bd is None: return "기준 날짜 대비 변화 속도를 계산할 수 없습니다."
        return (f"[기준 날짜 대비 변화 속도]\n기준: {bd['baseline_date']} ({bd['baseline_avg_total_change']}%)\n"
                f"선택: {bd['date']} ({bd['avg_total_change']}%) / 속도: {bd['baseline_change_speed_per_day']}%p/day")
    if rtype == "trend_change_speed":
        t = retrieved["trend"]
        if t is None: return "추세 계산 불가"
        return f"[추세] {t['trend_change_speed_per_day']}%p/day ({t['trend_direction']}) / {t['start_date']}~{t['end_date']}"
    found = retrieved.get("records", [])
    if not found: return "검색된 KG record가 없습니다."
    records_text = "\n\n".join([f"[결과 {i}]\n{record_to_text(r)}" for i, r in enumerate(found, 1)])
    if rtype in ["date_range_top_change", "date_range_low_change", "date_range_search"]:
        return f"[날짜 범위: {retrieved['start'].strftime('%Y-%m-%d')} ~ {retrieved['end'].strftime('%Y-%m-%d')}]\n{records_text}"
    if rtype == "date_search":
        return f"[날짜: {retrieved['start'].strftime('%Y-%m-%d')}]\n{records_text}"
    return f"[검색 결과 유형: {rtype}]\n{records_text}"
 
 
def make_answer_hint(retrieved):
    rtype = retrieved.get("type")
    if rtype == "summary":
        s = retrieved["summary"]
        max_r, min_r = s["max_change_record"], s["min_change_record"]
        return (f"[정답 후보] 총 {s['total']}개 / 평균 total_change: {s['avg_total']:.4f}%\n"
                f"최대: {max_r['target_image']} ({max_r['date']}, {max_r['total_change']}%)\n"
                f"최소: {min_r['target_image']} ({min_r['date']}, {min_r['total_change']}%)")
    if rtype in ["two_date_total_change_diff", "two_date_interval_speed"]:
        comp = retrieved["comparison"]
        if comp["diff"] is None: return f"[정답 후보] {comp['date1']}과 {comp['date2']} 계산 불가."
        d1, d2 = comp["date1_info"], comp["date2_info"]
        return f"[정답 후보] {comp['date1']}: {d1['avg_total_change']}% / {comp['date2']}: {d2['avg_total_change']}% / 차이: {comp['diff']}%p ({comp['direction']})"
    if rtype in ["top_baseline_change_speed", "low_baseline_change_speed"]:
        bd = retrieved["best_date"]
        if bd is None: return "[정답 후보] 계산 불가."
        return f"[정답 후보] {bd['date']}: {bd['avg_total_change']}% / 속도: {bd['baseline_change_speed_per_day']}%p/day"
    if rtype == "trend_change_speed":
        t = retrieved["trend"]
        return f"[정답 후보] 추세: {t['trend_change_speed_per_day']}%p/day ({t['trend_direction']})" if t else "[정답 후보] 추세 계산 불가."
    found = retrieved.get("records", [])
    if not found: return "[정답 후보] 조건에 맞는 record 없음."
    best = found[0]
    target_image = best.get('target_image') or best.get('filename', 'unknown')
    if rtype in ["date_range_top_change", "date_range_low_change"]:
        s, e = retrieved["start"].strftime("%Y-%m-%d"), retrieved["end"].strftime("%Y-%m-%d")
        return f"[정답 후보] {s}~{e} 중: {target_image} ({best['date']}) / total_change: {best['total_change']}%"
    return f"[정답 후보] {target_image} ({best['date']}) / total_change: {best['total_change']}% / {best.get('change_level')}"

## KG Json 로드 + Qwen 로드 + 질의응답 횟수

In [11]:
# Cell 9: LLM 로드 (notebook 버전 - Qwen)
# =============================================================
 
qwen_model = None
qwen_tokenizer = None
qwen_device = device
 
try:
    qwen_model_name = "meta-llama/Llama-3.1-8B-Instruct"
    qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
    qwen_model = AutoModelForCausalLM.from_pretrained(
        qwen_model_name,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    ).eval()
    print("Qwen LLM 로드 완료")
except Exception as e:
    print("Qwen LLM 로드 실패:", e)
 
 
def generate_with_llm(messages, max_new_tokens=256):
    if qwen_model is None or qwen_tokenizer is None:
        return "LLM 로드 실패로 응답을 생성할 수 없습니다."
    text = qwen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = qwen_tokenizer(text, return_tensors="pt").to(qwen_device)
    with torch.no_grad():
        outputs = qwen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=qwen_tokenizer.eos_token_id)
    return qwen_tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
 
 
def ask_llm(question, context, answer_hint):
    system = """너는 이미지 변화 탐지 KG 결과를 설명하는 도우미다.
[정답 후보]와 [KG context]에 있는 값만 사용한다. 없는 정보는 추측하지 않는다.
- 변화율 = total_change = change + strong_change
- 수치를 임의로 계산하거나 바꾸지 않는다.
- 표, 목록 없이 자연스러운 한국어 문장 3~5문장으로 답한다.
- KG context에 해당 날짜 데이터가 없으면 절대 추측하지 않는다.
- daily_change_speed_per_day가 양수면 반드시 증가로 설명한다.
- previous_date 값을 절대 바꾸지 않는다."""
    user = f"[질문]\n{question}\n\n{answer_hint}\n\n[KG context]\n{context}\n\n위 정보로 답변해줘."
    return generate_with_llm(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        max_new_tokens=256
    )
 
 
# =============================================================
# Cell 10: Agent State 정의 (total_plus 버전)
# =============================================================
 
class AgentState(TypedDict):
    question: str
    image_path: Optional[str]
    route: Optional[str]
    route_reason: Optional[str]
    image_result: Optional[dict]
    kg_result: Optional[str]
    report_result: Optional[str]
    final_answer: Optional[str]
    mask_path: Optional[str]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Qwen LLM 로드 완료


In [12]:
!pip install neo4j

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 6.1 MB/s eta 0:00:00a 0:00:01


In [ ]:
from neo4j import GraphDatabase

# Neo4j 접속 정보는 코드에 직접 적지 않고 Kaggle Secrets(Add-ons > Secrets)에서 불러온다.
# 필요한 Secrets: NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD
URI = _get_secret("NEO4J_URI")
AUTH = (_get_secret("NEO4J_USER"), _get_secret("NEO4J_PASSWORD"))

if not URI or not AUTH[0] or not AUTH[1]:
    raise RuntimeError(
        "NEO4J_URI / NEO4J_USER / NEO4J_PASSWORD가 설정되지 않았습니다. "
        "Kaggle 노트북의 Add-ons > Secrets에 등록하세요."
    )

# 연결 테스트 코드
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()
    print("Neo4j 연결에 성공했습니다.")

def get_driver():
    global driver
    try:
        driver.verify_connectivity()
    except Exception:
        driver = GraphDatabase.driver(URI, auth=AUTH)
    return driver

# 사용할 때
with get_driver().session() as session:
    result = session.run("MATCH (n) RETURN count(n) as cnt")
    print(result.single()["cnt"])

In [14]:

# # ============================================================
# # Cell 2: KG JSON 로드
# # ============================================================
# KG_GRAPH_PATH = Path("/kaggle/working/full_302_change_event_kg_graph_image.json")

# with open(KG_GRAPH_PATH, "r", encoding="utf-8") as f:
#     kg_data = json.load(f)

# records_data = kg_data.get("records", [])
# nodes_data   = kg_data.get("nodes", {})
# edges_data   = kg_data.get("edges", [])

# print(f"records: {len(records_data)}개")
# print(f"nodes:   {len(nodes_data)}개")
# print(f"edges:   {len(edges_data)}개")


## LangChain Tools

In [15]:
@tool
def dinov3_tool(image_path: str) -> Dict[str, Any]:
    """입력 이미지 1장을 DINOv3 모델로 분석하여 예측 마스크 이미지와 기본 변화 비율을 생성한다."""
    return infer_single_image_with_dinov3(
        image_path=image_path,
        save_dir="/kaggle/working/agent_outputs",
        change_threshold=0.35
    )
 
 
@tool
def ratio_tool(single_result: Dict[str, Any]) -> Dict[str, Any]:
    """DINOv3 결과에서 변화 비율을 추출하고 정리한다."""
    filename = single_result.get("filename") or "unknown_image"
    timestamp = single_result.get("timestamp") or "unknown_time"
    mask_path = single_result.get("mask_path")
    no_change = round(float(single_result.get("no_change", 0.0)), 2)
    change = round(float(single_result.get("change", 0.0)), 2)
    strong_change = round(float(single_result.get("strong_change", 0.0)), 2)
    total_change = round(float(single_result.get("total_change", change + strong_change)), 2)
    return {"filename": filename, 
            "timestamp": timestamp, 
            "mask_path": mask_path,
            "no_change": no_change, 
            "change": change, 
            "strong_change": strong_change, 
            "total_change": total_change}

 
@tool
def validation_tool(ratio_result: Dict[str, Any]) -> Dict[str, Any]:
    """KG 저장 전 데이터 무결성을 검증한다."""
    errors = []
    no_change = float(ratio_result.get("no_change", 0.0))
    change = float(ratio_result.get("change", 0.0))
    strong_change = float(ratio_result.get("strong_change", 0.0))
    total_change = float(ratio_result.get("total_change", 0.0))
    filename = ratio_result.get("filename")
    timestamp = ratio_result.get("timestamp")
    mask_path = ratio_result.get("mask_path")
 
    if total_change < 0: errors.append("total_change가 음수입니다.")
    if strong_change > total_change: 
        errors.append("strong_change가 total_change보다 큽니다.")
    for field_name, value in {"filename": filename, "timestamp": timestamp, 
                              "mask_path": mask_path}.items():
        if value is None or str(value).strip() == "": errors.append(f"{field_name} 누락")
    expected_total = round(change + strong_change, 2)
    if abs(total_change - expected_total) > 0.01:
        errors.append(f"total_change 계산 불일치 (expected={expected_total}, actual={total_change})")
    if mask_path and not os.path.exists(mask_path):
        errors.append(f"mask 파일 없음: {mask_path}")
 
    if len(errors) == 0:
        print("\n✅ [Validation 성공]")
        print(f"파일명: {filename} / total_change: {total_change:.2f}%")
    else:
        print("\n❌ [Validation 실패]")
        for err in errors: print(f" - {err}")
 
    return {"is_valid": len(errors) == 0, "errors": errors, "ratio_result": ratio_result}
 
 
@tool
def rag_tool(ratio_result: Dict[str, Any]) -> Dict[str, Any]:
    """변화 비율을 기준으로 RAG 문서를 검색한다."""
    img_file = ratio_result.get("filename", "unknown_image")
    no_change = float(ratio_result.get("no_change", 0.0))
    change = float(ratio_result.get("change", 0.0))
    strong_change = float(ratio_result.get("strong_change", 0.0))
    total_change = float(ratio_result.get("total_change", change + strong_change))
 
    total_doc, total_candidates = retrieve_ratio_doc(value=total_change, category="total_change")
    strong_doc, strong_candidates = retrieve_ratio_doc(value=strong_change, category="strong_change")
 
    rag_context = make_rag_context(
        img_file=img_file, no_change=no_change, change=change,
        strong_change=strong_change, total_change=total_change,
        total_doc=total_doc, strong_doc=strong_doc
    )
 
    return {"filename": img_file, "total_doc": total_doc, "strong_doc": strong_doc,
            "total_candidates": total_candidates[:3], "strong_candidates": strong_candidates[:3],
            "rag_context": rag_context}
 
 
@tool
def vlm_tool(ratio_result: Dict[str, Any], rag_result: Dict[str, Any]) -> Dict[str, Any]:
    """마스크 이미지와 RAG context를 이용해 VLM 기반 시각적 분포 설명을 생성한다."""
    mask_path = ratio_result.get("mask_path")
    rag_context = rag_result.get("rag_context", "")
 
    if mask_path is None:
        return {"vlm_raw_output": "VLM skipped",
                "vlm_report": "마스크 이미지 경로가 없어 VLM 관찰 보고서는 생성하지 못했습니다."}
    try:
        response = run_vlm(mask_image_path=mask_path, rag_context=rag_context)
        vlm_report = clean_vlm_report(response)
    except Exception as e:
        print("VLM 추론 실패:", e)
        response = "VLM failed"
        vlm_report = "VLM 추론 오류로 마스크 이미지 기반 관찰 보고서는 생성하지 못했습니다."
 
    return {"vlm_raw_output": response, "vlm_report": vlm_report}
 
 
@tool
def kg_append_tool(ratio_result: Dict[str, Any], rag_result: Dict[str, Any], vlm_result: Dict[str, Any]) -> Dict[str, Any]:
    """DINOv3, Ratio, RAG, VLM 결과를 Neo4j KG에 직접 저장한다."""
    
    img_file      = ratio_result.get("filename", "unknown_image")
    mask_path     = ratio_result.get("mask_path")
    no_change     = float(ratio_result.get("no_change", 0.0))
    change        = float(ratio_result.get("change", 0.0))
    strong_change = float(ratio_result.get("strong_change", 0.0))
    total_change  = round(change + strong_change, 2)
    total_doc     = rag_result.get("total_doc")
    strong_doc    = rag_result.get("strong_doc")
    vlm_report    = vlm_result.get("vlm_report", "")
    timestamp     = extract_timestamp(img_file)
    event_id      = f"ChangeEvent_{img_file.replace('.png', '')}"

    # progress 생성
    progress = (
        f"입력 이미지는 No Change {no_change:.2f}%, Change {change:.2f}%, Strong Change {strong_change:.2f}%로 분석되었습니다. "
        f"전체 변화율은 {total_change:.2f}%이며, RAG 기준으로 '{total_doc['level']}' 수준입니다. "
        f"강한 변화율은 {strong_change:.2f}%이며, '{strong_doc['level']}' 수준입니다."
    )

    # Neo4j 직접 저장
    with get_driver().session() as session:

        # Record 노드
        session.run("""
            MERGE (n:Record {change_event_id: $id})
            SET n.target_image        = $target_image,
                n.date                = $date,
                n.change              = $change,
                n.strong_change       = $strong_change,
                n.total_change        = $total_change,
                n.change_level        = $change_level,
                n.strong_change_level = $strong_change_level,
                n.vlm_report          = $vlm_report,
                n.progress            = $progress
        """,
            id=event_id,
            target_image=img_file,
            date=timestamp,
            change=change,
            strong_change=strong_change,
            total_change=total_change,
            change_level=total_doc["level"],
            strong_change_level=strong_doc["level"],
            vlm_report=vlm_report[:5000],
            progress=progress[:5000]
        )

        # ChangeEvent 노드
        session.run("""
            MERGE (n:ChangeEvent {node_id: $node_id})
            SET n.change_event_id = $event_id,
                n.change          = $change,
                n.strong_change   = $strong_change,
                n.total_change    = $total_change
        """,
            node_id=event_id,
            event_id=event_id,
            change=change,
            strong_change=strong_change,
            total_change=total_change
        )

        # Image 노드
        image_node_id = f"Image_{img_file.replace('.png', '')}"
        session.run("""
            MERGE (n:Image {node_id: $node_id})
            SET n.filename = $filename
        """, node_id=image_node_id, filename=img_file)

        # Date 노드
        date_node_id = f"Date_{timestamp.replace('-', '_')}"
        session.run("""
            MERGE (n:Date {node_id: $node_id})
            SET n.date = $date
        """, node_id=date_node_id, date=timestamp)

        # ChangeLevel 노드
        cl_node_id = f"ChangeLevel_{total_doc['level'].replace(' ', '_')}"
        session.run("""
            MERGE (n:ChangeLevel {node_id: $node_id})
            SET n.label = $label
        """, node_id=cl_node_id, label=total_doc["level"])

        # StrongChangeLevel 노드
        scl_node_id = f"StrongChangeLevel_{strong_doc['level'].replace(' ', '_')}"
        session.run("""
            MERGE (n:StrongChangeLevel {node_id: $node_id})
            SET n.label = $label
        """, node_id=scl_node_id, label=strong_doc["level"])

        # 엣지 생성
        for source, target, relation in [
            (event_id,    image_node_id, "TARGET_IMAGE"),
            (event_id,    date_node_id,  "OCCURRED_ON"),
            (event_id,    cl_node_id,    "HAS_CHANGE_LEVEL"),
            (event_id,    scl_node_id,   "HAS_STRONG_CHANGE_LEVEL"),
        ]:
            session.run(f"""
                MATCH (a {{node_id: $source}})
                MATCH (b {{node_id: $target}})
                MERGE (a)-[:{relation}]->(b)
            """, source=source, target=target)

        print(f"✅ Neo4j 직접 저장 완료: {event_id}")

    # refresh_kg_records도 Neo4j 기반으로
    refresh_kg_records()

    return {
        "event_id":  event_id,
        "final_output": {
            "timestamp":           timestamp,
            "filename":            img_file,
            "total_change_level":  total_doc["level"],
            "strong_change_level": strong_doc["level"],
            "ratios": {
                "no_change":    no_change,
                "change":       change,
                "strong_change": strong_change,
                "total_change": total_change
            },
            "progress":    progress,
            "distribution": vlm_report,
            "mask_path":   mask_path
        },
        "answer": (
            f"{progress} "
            f"{vlm_report} "
            f"결과는 KG '{event_id}'로 저장되었습니다."
        ),
        "mask_path": mask_path
    }

In [16]:
# Cell 12: KG Agent Tools (total_plus 버전)
# ===========================================================
 
@tool
def kg_retrieve_tool(question: str) -> Dict[str, Any]:
    """Neo4j Graph DB에서 KG record를 검색한다."""
    retrieved = neo4j_retrieve_records(question, top_k=5)
    
    with get_driver().session() as session:
        count = session.run("MATCH (n:Record) RETURN count(n) as cnt").single()["cnt"]
    
    return {
        "question": question,
        "retrieved": retrieved,
        "num_records": count
    }

 
 
@tool
def kg_context_tool(retrieve_result: Dict[str, Any]) -> Dict[str, Any]:
    """검색된 KG 결과를 LLM이 읽을 수 있는 context와 answer hint로 변환한다."""
    question = retrieve_result.get("question", "")
    retrieved = retrieve_result.get("retrieved", {})
    try: context = build_context(retrieved)
    except Exception as e: context = f"KG context 생성 중 오류: {str(e)}"
    try: answer_hint = make_answer_hint(retrieved)
    except Exception as e: answer_hint = f"정답 후보 생성 중 오류: {str(e)}"
    return {"question": question, "retrieved": retrieved, "context": context, "answer_hint": answer_hint}
 
 
@tool
def kg_answer_tool(context_result: Dict[str, Any]) -> Dict[str, Any]:
    """KG context와 answer hint를 바탕으로 LLM이 자연어 답변을 생성한다."""
    question = context_result.get("question", "")
    context = context_result.get("context", "")
    answer_hint = context_result.get("answer_hint", "")
    answer = ask_llm(question=question, context=context, answer_hint=answer_hint)
    return {"answer": answer, "context": context, "answer_hint": answer_hint}
 

In [17]:
# =============================================================
# Cell 13: Report Agent Tools (total_plus 버전)
# =============================================================
 
@tool
def report_summary_tool(request: str) -> Dict[str, Any]:
    """Neo4j KG 기반 전체 통계 요약"""
    with get_driver().session() as session:
        result = session.run("""
            MATCH (n:Record)
            RETURN 
                count(n) as total,
                avg(n.total_change) as avg_total,
                avg(n.strong_change) as avg_strong,
                max(n.total_change) as max_total,
                min(n.total_change) as min_total
        """)
        stats = result.single()

        max_r = session.run("""
            MATCH (n:Record)
            RETURN n ORDER BY n.total_change DESC LIMIT 1
        """).single()["n"]

        min_r = session.run("""
            MATCH (n:Record)
            RETURN n ORDER BY n.total_change ASC LIMIT 1
        """).single()["n"]

        all_records = [
            dict(r["n"]) for r in session.run("MATCH (n:Record) RETURN n")
        ]

    trend = calculate_trend_change_speed(all_records)
    trend_text = f"{trend['trend_change_speed_per_day']}%p/day ({trend['trend_direction']})" if trend else "계산 불가"

    summary = {
        "num_records":      int(stats["total"]),
        "avg_total_change": round(float(stats["avg_total"] or 0), 2),
        "avg_strong_change": round(float(stats["avg_strong"] or 0), 2),
        "max_total_image":  dict(max_r).get("target_image"),
        "max_total_change": round(float(dict(max_r).get("total_change", 0)), 2),
        "min_total_image":  dict(min_r).get("target_image"),
        "min_total_change": round(float(dict(min_r).get("total_change", 0)), 2),
        "trend": trend
    }

    return {"request": request, "num_records": summary["num_records"], "summary": summary, "records": all_records}
 
@tool
def report_answer_tool(summary_result: Dict[str, Any]) -> Dict[str, Any]:
    """통계 요약을 바탕으로 LLM이 종합 보고서를 작성한다."""
    request = summary_result.get("request", "전체 보고서 작성")
    summary = summary_result.get("summary", {})
    num_records = summary_result.get("num_records", 0)
 
    if num_records == 0:
        return {"answer": "KG records가 없어 보고서를 생성할 수 없습니다."}
 
    trend = summary.get("trend", {})
    trend_text = f"{trend.get('trend_change_speed_per_day', 'N/A')}%p/day ({trend.get('trend_direction', 'N/A')})" if trend else "계산 불가"
 
    context = (
        f"[전체 요약]\n"
        f"총 분석 이미지: {num_records}개\n"
        f"평균 total_change: {summary.get('avg_total_change', 'N/A')}%\n"
        f"평균 strong_change: {summary.get('avg_strong_change', 'N/A')}%\n"
        f"최대 변화 이미지: {summary.get('max_total_image', 'N/A')} ({summary.get('max_total_change', 'N/A')}%)\n"
        f"최소 변화 이미지: {summary.get('min_total_image', 'N/A')} ({summary.get('min_total_change', 'N/A')}%)\n"
        f"전체 추세: {trend_text}"
    )
 
    answer = ask_llm(
        question=request,
        context=context,
        answer_hint="위 데이터를 바탕으로 전체 결정화 진행 보고서를 작성해줘."
    )
 
    return {"answer": answer, "context": context}
 

In [18]:
# Cell 14: Supervisor Agent (total_plus 버전 - LLM 라우팅)
# =============================================================
 
def supervisor_agent(state: AgentState) -> AgentState:
    """LLM 기반 Supervisor - IMAGE_AGENT / KG_AGENT / REPORT_AGENT 라우팅"""
    question = state["question"]
    image_path = state.get("image_path")
    has_image = image_path is not None and str(image_path).strip() != ""
 
    prompt = f"""
너는 단백질 합성 이미지 분석 시스템의 Supervisor Agent이다.
사용자 입력을 보고 어떤 하위 에이전트를 실행할지 선택하는 것이 너의 역할이다.
 
[사용자 질문]
{question}
 
[이미지 입력 여부]
{has_image}
 
[하위 에이전트 목록]
1. IMAGE_AGENT - 이미지 업로드 시 / 새 이미지 분석 요청 시
2. KG_AGENT - 기존 KG 데이터에서 특정 날짜/이미지/변화율 질문 시
3. REPORT_AGENT - 전체 요약/보고서/통계/추세 요청 시
 
[판단 규칙]
- 이미지가 입력되어 있으면 IMAGE_AGENT를 선택한다.
- 이미지가 없고 특정 날짜/이미지/변화율/변화 속도를 묻는다면 KG_AGENT를 선택한다.
- 이미지가 없고 전체 요약/보고서/통계/추세를 요청하면 REPORT_AGENT를 선택한다.
- 반드시 JSON 하나만 출력한다.
 
반드시 아래 형식 중 하나로만 답하라.
{{"route": "IMAGE_AGENT", "reason": "..."}}
{{"route": "KG_AGENT", "reason": "..."}}
{{"route": "REPORT_AGENT", "reason": "..."}}
"""
 
    generated_text = generate_with_llm([{"role": "user", "content": prompt}], max_new_tokens=100).strip()
    print(f"\n[Supervisor raw output]\n{generated_text}")
 
    route = None
    reason = ""
    match = re.search(r'\{.*?\}', generated_text, re.DOTALL)
    if match:
        try:
            parsed = json.loads(match.group())
            route = parsed.get("route")
            reason = parsed.get("reason", "")
        except:
            pass
 
    # 안전장치: LLM 출력 이상 시 규칙 기반 보정
    if route not in ["IMAGE_AGENT", "KG_AGENT", "REPORT_AGENT"]:
        if has_image:
            route = "IMAGE_AGENT"
            reason = "이미지 입력 감지 (규칙 기반 보정)"
        elif any(w in question for w in ["보고서", "요약", "추세", "전체 분석", "전체적으로", "통계"]):
            route = "REPORT_AGENT"
            reason = "보고서/요약 키워드 감지 (규칙 기반 보정)"
        else:
            route = "KG_AGENT"
            reason = "기본값 KG_AGENT (규칙 기반 보정)"
 
    print(f"[Supervisor] route → {route} | reason: {reason}")
    return {**state, "route": route, "route_reason": reason}

## LangGraph 다중 에이전트

In [19]:
# =============================================================
# Cell 15: Agent Node 함수들
# =============================================================
 
def image_agent_node(state: AgentState) -> AgentState:
    print(f"[ImageAgent] 시작: {state.get('image_path')}")
    try:
        # 1. DINOv3 추론
        single_result = dinov3_tool.invoke({"image_path": state["image_path"]})
 
        # 2. 비율 정리
        ratio_result = ratio_tool.invoke({"single_result": single_result})
 
        # 3. Validation
        val_result = validation_tool.invoke({"ratio_result": ratio_result})
        if not val_result["is_valid"]:
            return {**state, "final_answer": f"Validation 실패: {val_result['errors']}", "mask_path": None}
 
        # 4. RAG 검색
        rag_result = rag_tool.invoke({"ratio_result": ratio_result})
 
        # 5. VLM 분석
        vlm_result = vlm_tool.invoke({"ratio_result": ratio_result, "rag_result": rag_result})
 
        # 6. KG 저장
        kg_result = kg_append_tool.invoke({"ratio_result": ratio_result, "rag_result": rag_result, "vlm_result": vlm_result})
 
        answer = kg_result.get("answer", "")
        mask_path = kg_result.get("mask_path")
 
        print(f"[ImageAgent] 완료")
        return {**state, "image_result": kg_result.get("final_output"), "final_answer": answer, "mask_path": mask_path}
    except Exception as e:
        return {**state, "final_answer": f"[ImageAgent] 오류: {str(e)}", "mask_path": None}
 
 
def kg_agent_node(state: AgentState) -> AgentState:
    question = state.get("question", "")
    image_result = state.get("image_result")
    if image_result and any(w in question for w in ["방금", "이 이미지", "업로드한", "유사한"]):
        total_change = image_result["ratios"]["total_change"]
        question = f"{question} (참고: 방금 분석한 이미지의 total_change는 {total_change}%)"
    print(f"[KGAgent] 시작: {question}")
    try:
        retrieve_result = kg_retrieve_tool.invoke({"question": question})
        context_result = kg_context_tool.invoke({"retrieve_result": retrieve_result})
        answer_result = kg_answer_tool.invoke({"context_result": context_result})
        answer = answer_result.get("answer", "KG 질의응답 결과가 없습니다.")
        print("[KGAgent] 완료")
        return {**state, "kg_result": answer, "final_answer": answer}
    except Exception as e:
        return {**state, "final_answer": f"[KGAgent] 오류: {str(e)}"}
 
 
def report_agent_node(state: AgentState) -> AgentState:
    print("[ReportAgent] 시작")
    try:
        summary_result = report_summary_tool.invoke({"request": state.get("question", "전체 보고서 작성")})
        answer_result = report_answer_tool.invoke({"summary_result": summary_result})
        answer = answer_result.get("answer", "보고서 생성 결과가 없습니다.")
        print("[ReportAgent] 완료")
        return {**state, "report_result": answer, "final_answer": answer}
    except Exception as e:
        return {**state, "final_answer": f"[ReportAgent] 오류: {str(e)}"}
 
 
def route_decision(state: AgentState):
    return state["route"]
 

In [29]:
# ============================================================
# Cell 16 마지막 부분 + 이후 코드 - 아래 내용을 노트북에 추가하세요
# ============================================================

def build_graph():
    graph = StateGraph(AgentState)
    graph.add_node("supervisor",    supervisor_agent)
    graph.add_node("image_agent",   image_agent_node)
    graph.add_node("kg_agent",      kg_agent_node)
    graph.add_node("report_agent",  report_agent_node)
    graph.set_entry_point("supervisor")
    graph.add_conditional_edges("supervisor", route_decision, {
        "IMAGE_AGENT":  "image_agent",
        "KG_AGENT":     "kg_agent",
        "REPORT_AGENT": "report_agent"
    })
    graph.add_edge("image_agent",  END)
    graph.add_edge("kg_agent",     END)
    graph.add_edge("report_agent", END)
    return graph.compile()

app = build_graph()


def ai_agent(question: str, image_path: str = None) -> dict:
    result = app.invoke({
        "question": question, "image_path": image_path,
        "route": None, "route_reason": None,
        "image_result": None, "kg_result": None, "report_result": None,
        "final_answer": None, "mask_path": None
    })
    print("\n========== 최종 응답 ==========")
    print(result.get("final_answer", "응답을 생성하지 못했습니다."))
    print("================================\n")
    return result


# ============================================================
# Cell 17: KG 로드
# ============================================================

if KG_GRAPH_PATH.exists():
    refresh_kg_records()
    print(f"KG 로드 완료: {len(records)}개 records")
else:
    print("KG JSON 없음 - 새 이미지 분석 후 자동 생성됩니다.")

[KG] records 갱신 완료: 66개
KG 로드 완료: 66개 records


In [30]:
!pip install neo4j fastapi uvicorn pyngrok -q

In [33]:
from pyngrok import ngrok

ngrok.kill()

public_url = ngrok.connect(8000)
print(public_url)

NgrokTunnel: "https://balance-defensive-yoga.ngrok-free.dev" -> "http://localhost:8000"


In [ ]:
from pyngrok import ngrok

# ngrok authtoken은 Kaggle Secrets(Add-ons > Secrets)의 NGROK_AUTHTOKEN에서 불러온다.
ngrok.set_auth_token(_get_secret("NGROK_AUTHTOKEN"))

# 기존에 작성하셨던 터널링 코드 (예: FastAPI 기본 포트인 8000번)
public_url = ngrok.connect(8000)
print("외부 접속 주소:", public_url)

In [ ]:
from fastapi import FastAPI, HTTPException, Header, Depends
from pyngrok import ngrok
import uvicorn
import threading
import logging
import time

# 모든 기존 터널 종료 후 재시작
ngrok.kill()
time.sleep(2)
ngrok.set_auth_token(_get_secret("NGROK_AUTHTOKEN"))

# 로컬 orchestrator(config.py의 KAGGLE_API_KEY, .env)와 반드시 같은 값을 써야 한다.
API_SHARED_KEY = _get_secret("KAGGLE_API_KEY")
if not API_SHARED_KEY:
    raise RuntimeError(
        "KAGGLE_API_KEY가 설정되지 않았습니다. Kaggle 노트북의 Add-ons > Secrets에 등록하세요. "
        "로컬 .env의 KAGGLE_API_KEY와 반드시 같은 값이어야 합니다."
    )


def verify_api_key(x_api_key: str = Header(default=None)):
    if x_api_key != API_SHARED_KEY:
        raise HTTPException(status_code=401, detail="Invalid or missing API key")


app = FastAPI()


@app.post("/analyze", dependencies=[Depends(verify_api_key)])
async def analyze(image_path: str):
    try:
        # 1. DINOv3
        single_result = dinov3_tool.invoke({"image_path": image_path})

        # 2. ratio
        ratio_result = ratio_tool.invoke({"single_result": single_result})

        # 3. validation
        val_result = validation_tool.invoke({"ratio_result": ratio_result})
        if not val_result["is_valid"]:
            return {"error": f"Validation 실패: {val_result['errors']}"}

        # 4. RAG
        rag_result = rag_tool.invoke({"ratio_result": ratio_result})

        # 5. VLM
        vlm_result = vlm_tool.invoke({"ratio_result": ratio_result, "rag_result": rag_result})

        # 6. KG 저장
        kg_result = kg_append_tool.invoke({
            "ratio_result": ratio_result,
            "rag_result": rag_result,
            "vlm_result": vlm_result
        })

        return {
            "filename":            ratio_result["filename"],
            "total_change":        ratio_result["total_change"],
            "change_level":        rag_result["total_doc"]["level"],
            "strong_change_level": rag_result["strong_doc"]["level"],
            "vlm_report":          vlm_result["vlm_report"],
            "event_id":            kg_result["event_id"]
        }
    except HTTPException:
        raise # Validation 에러는 그대로 통과
    except Exception as e:
        logging.error(f"파이프라인 실행 중 치명적 에러 발생: {str(e)}")
        raise HTTPException(status_code=500, detail=f"서버 내부 에러: {str(e)}")


@app.post("/search_kg", dependencies=[Depends(verify_api_key)])
async def search_kg(question: str):
    result = neo4j_retrieve_records(question)
    return result


@app.post("/report", dependencies=[Depends(verify_api_key)])
async def report():
    result = report_summary_tool.invoke({"request": "전체 보고서"})
    return result


@app.post("/rag_search", dependencies=[Depends(verify_api_key)])
async def rag_search(query: str):
    try:
        total_doc, _ = retrieve_ratio_doc(value=float(query), category="total_change")
        strong_doc, _ = retrieve_ratio_doc(value=float(query), category="strong_change")
        return {"result": f"{total_doc['text']}\n{strong_doc['text']}"}
    except Exception as e:
        return {"error": str(e)}


# ngrok으로 외부 노출 (터널은 하나만 연다)
ngrok_tunnel = ngrok.connect(8000)
print(f"Public URL: {ngrok_tunnel.public_url}")
print("   -> 이 값을 로컬 .env의 KAGGLE_API_URL에 넣으세요.")

# 백그라운드 실행
thread = threading.Thread(
    target=uvicorn.run,
    args=(app,),
    kwargs={"host": "0.0.0.0", "port": 8000}
)
thread.daemon = True
thread.start()

In [23]:
# 현재 GPU 사용량 확인
print(torch.cuda.memory_summary(device=None, abbreviated=False))

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |  13192 MiB |  13192 MiB | 336988 MiB | 323795 MiB |
|       from large pool |  13090 MiB |  13090 MiB | 335332 MiB | 322242 MiB |
|       from small pool |    102 MiB |    102 MiB |   1655 MiB |   1553 MiB |
|---------------------------------------------------------------------------|
| Active memory         |  13192 MiB |  13192 MiB | 336988 MiB | 323795 MiB |
|       from large pool |  13090 MiB |  13090 MiB | 335332 MiB |

In [26]:
def safe_float(val):
    try:
        return float(val)
    except:
        return 0.0
 
 
def parse_date(date_str):
    return datetime.strptime(date_str, "%Y-%m-%d")
 
 
def parse_record_datetime(r):
    try:
        date_str = r.get("date") or r.get("timestamp")
        img_file = r.get("target_image") or r.get("filename", "")
        time_parts = img_file.replace(".png", "").split("_")
        time_str = time_parts[2] if len(time_parts) > 2 else "000000"
        return datetime.strptime(f"{date_str} {time_str}", "%Y-%m-%d %H%M%S")
    except:
        try:
            return parse_date(r.get("date") or r.get("timestamp", "2000-01-01"))
        except:
            return None
 
 
def classify_change_speed(speed):
    
    if speed is None: return "No Previous Data"
    if speed >= 5.0:   return "Rapid Increase"
    elif speed >= 1.0: return "Moderate Increase"
    elif speed >= 0.1: return "Slow Increase"
    elif speed > -0.1: return "Stable"
    elif speed > -1.0: return "Slow Decrease"
    elif speed > -5.0: return "Moderate Decrease"
    else:              return "Rapid Decrease"
 
 
def safe_label_id(text):
    return str(text).strip().replace(" ", "_").replace("/", "_").replace(":", "_")
 
 
def make_change_event_id(filename):     return f"ChangeEvent_{filename.replace('.png', '')}"
def make_image_node_id(filename):       return f"Image_{filename.replace('.png', '')}"
def make_reference_image_node_id(fn):   return f"ReferenceImage_{fn.replace('.png', '')}"
def make_date_node_id(date_str):        return "Date_" + date_str.replace("-", "_")
def make_change_level_node_id(level):   return f"ChangeLevel_{safe_label_id(level)}"
def make_strong_change_level_node_id(level): return f"StrongChangeLevel_{safe_label_id(level)}"
def make_speed_level_node_id(level):    return f"SpeedLevel_{safe_label_id(level)}"
 
 
def add_node(nodes_dict, node_id, node_type, properties=None):
    if properties is None: properties = {}
    if node_id not in nodes_dict:
        nodes_dict[node_id] = {"id": node_id, "type": node_type, "properties": properties}
    else:
        nodes_dict[node_id]["properties"].update(properties)
 
 
def add_edge(edges, source, target, relation, properties=None):
    edges.append({"source": source, "target": target, "relation": relation, "properties": properties or {}})
 
 
def build_final_result(img_file, mask_path, no_change, change, strong_change, total_doc, strong_doc, vlm_report):
    total_change = round(change + strong_change, 2)
    timestamp = extract_timestamp(img_file)
    progress = (
        f"입력 이미지는 No Change {no_change:.2f}%, Change {change:.2f}%, Strong Change {strong_change:.2f}%로 분석되었습니다. "
        f"전체 변화율은 {total_change:.2f}%이며, RAG 기준으로 '{total_doc['level']}' 수준입니다. {total_doc['text']} "
        f"강한 변화율은 {strong_change:.2f}%이며, '{strong_doc['level']}' 수준입니다. {strong_doc['text']}\n\n[VLM 분석]\n{vlm_report}"
    )
    return {
        "timestamp": timestamp, "filename": img_file,
        "total_change_level": total_doc["level"], "strong_change_level": strong_doc["level"],
        "ratios": {"no_change": no_change, "change": change, "strong_change": strong_change, "total_change": total_change},
        "progress": progress, "distribution": vlm_report, "mask_path": mask_path
    }
 
 
records = []
 
 
def refresh_kg_records():
    global records
    with get_driver().session() as session:
        result = session.run("MATCH (n:Record) RETURN n")
        records = [dict(r["n"]) for r in result]
    print(f"[KG] records 갱신 완료: {len(records)}개")
 
 
def append_final_output_to_kg_graph(final_output):
    img_file = final_output["filename"]
    date = final_output["timestamp"]
    ratios = final_output["ratios"]
 
    kg_data = {"nodes": {}, "edges": [], "records": []}
    if KG_GRAPH_PATH.exists():
        with open(KG_GRAPH_PATH, "r", encoding="utf-8") as f:
            kg_data = json.load(f)
 
    nodes_dict = {n["id"]: n for n in kg_data.get("nodes", {}).values()} if isinstance(kg_data.get("nodes"), dict) else {}
    edges = kg_data.get("edges", [])
    flat_records = kg_data.get("records", [])
 
    # 날짜별 평균 계산 (기존 records 포함)
    date_to_values = defaultdict(list)
    for r in flat_records:
        date_to_values[r["date"]].append(float(r["total_change"]))
    date_to_values[date].append(float(ratios["total_change"]))
 
    date_avg = {d: round(sum(v)/len(v), 2) for d, v in date_to_values.items()}
    sorted_dates = sorted(date_avg.keys(), key=lambda d: parse_date(d))
    prev_date = None
    date_rel = {}
    for cur in sorted_dates:
        if prev_date is None:
            date_rel[cur] = {"previous_date": None, "previous_avg_total_change": None,
                             "days_from_previous": None, "daily_change_delta": None,
                             "daily_change_speed_per_day": None, "change_speed_level": "No Previous Data"}
        else:
            prev_avg = date_avg[prev_date]
            days = (parse_date(cur) - parse_date(prev_date)).days
            delta = round(date_avg[cur] - prev_avg, 2)
            speed = round(delta / days, 2) if days > 0 else None
            date_rel[cur] = {
                "previous_date": prev_date, "previous_avg_total_change": prev_avg,
                "days_from_previous": days, "daily_change_delta": delta,
                "daily_change_speed_per_day": speed,
                "change_speed_level": classify_change_speed(speed)
            }
        prev_date = cur
 
    rel = date_rel.get(date, {})
    event_id = make_change_event_id(img_file)
 
    new_record = {
        "change_event_id":        event_id,
        "reference_image":        "initial_source.png",
        "target_image":           img_file,
        "date":                   date,
        "change":                 ratios["change"],
        "strong_change":          ratios["strong_change"],
        "total_change":           ratios["total_change"],
        "change_level":           final_output["total_change_level"],
        "strong_change_level":    final_output["strong_change_level"],
        "daily_avg_total_change": date_avg[date],
        "vlm_report":             final_output.get("distribution", ""),   # 추가
        "progress":               final_output.get("progress", ""),        # 추가
        **rel
    }
 
    flat_records.append(new_record)
 
    # 노드 추가
    add_node(nodes_dict, event_id, "ChangeEvent", {
        "change_event_id":        event_id,
        "change":                 ratios["change"],
        "strong_change":          ratios["strong_change"],
        "total_change":           ratios["total_change"],
        "daily_avg_total_change": date_avg[date],
        "vlm_report":             final_output.get("distribution", ""),   # 추가
        "progress":               final_output.get("progress", ""),        # 추가
        **{k: v for k, v in rel.items()}
    })
    add_node(nodes_dict, make_image_node_id(img_file), "Image", {"filename": img_file})
    add_node(nodes_dict, make_date_node_id(date), "Date", {"date": date, "daily_avg_total_change": date_avg[date]})
    add_node(nodes_dict, make_change_level_node_id(final_output["total_change_level"]), "ChangeLevel", {"label": final_output["total_change_level"]})
    add_node(nodes_dict, make_strong_change_level_node_id(final_output["strong_change_level"]), "StrongChangeLevel", {"label": final_output["strong_change_level"]})
    add_node(nodes_dict, make_speed_level_node_id(rel.get("change_speed_level", "No Previous Data")), "SpeedLevel", {"label": rel.get("change_speed_level", "No Previous Data")})
 
    add_edge(edges, event_id, make_image_node_id(img_file), "TARGET_IMAGE")
    add_edge(edges, event_id, make_date_node_id(date), "OCCURRED_ON")
    add_edge(edges, event_id, make_change_level_node_id(final_output["total_change_level"]), "HAS_CHANGE_LEVEL")
    add_edge(edges, event_id, make_strong_change_level_node_id(final_output["strong_change_level"]), "HAS_STRONG_CHANGE_LEVEL")
 
    kg_data = {"nodes": nodes_dict, "edges": edges, "records": flat_records}
    with open(KG_GRAPH_PATH, "w", encoding="utf-8") as f:
        json.dump(kg_data, f, ensure_ascii=False, indent=2)
 
    print(f"[KG] {event_id} 추가 완료")
    return kg_data, event_id
 

In [27]:
# # ============================================================
# # Cell 18: 전체 이미지 일괄 추론 + VLM + KG 저장
# # ============================================================

# import os
# import json

# os.makedirs(OUTPUT_DIR, exist_ok=True)

# # 이미 추론된 결과가 있으면 로드, 없으면 새로 추론
# if os.path.exists(INFERENCE_RESULTS):
#     with open(INFERENCE_RESULTS, "r") as f:
#         inference_results = json.load(f)
#     print(f"기존 추론 결과 로드: {len(inference_results)}개")
# else:
#     # DINOv3 일괄 추론
#     inference_results = {}
#     for img_file in val_files:
#         img_path = os.path.join(RESIZED_IMAGE_DIR, img_file)
#         img = cv2.imread(img_path)
#         img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
#         img_tensor = transform(img).unsqueeze(0).to(device)

#         with torch.no_grad():
#             output = dinov3_model(img_tensor)
#             probs = torch.softmax(output, dim=1).squeeze(0)
#             pred_mask = probs.argmax(dim=0)
#             pred_mask[(pred_mask != 2) & (probs[1] > 0.35)] = 1
#             pred_mask = pred_mask.cpu().numpy()

#         pad = padding_info[img_file]
#         h, w = pred_mask.shape
#         if pad['top'] > 0:    pred_mask[:pad['top'], :] = 255
#         if pad['bottom'] > 0: pred_mask[h-pad['bottom']:, :] = 255
#         if pad['left'] > 0:   pred_mask[:, :pad['left']] = 255
#         if pad['right'] > 0:  pred_mask[:, w-pad['right']:] = 255

#         valid_pixels = (pred_mask != 255).sum()
#         no_change     = round(float((pred_mask == 0).sum() / valid_pixels * 100), 2)
#         change        = round(float((pred_mask == 1).sum() / valid_pixels * 100), 2)
#         strong_change = round(float((pred_mask == 2).sum() / valid_pixels * 100), 2)

#         inference_results[img_file] = {
#             'no_change': no_change,
#             'change': change,
#             'strong_change': strong_change
#         }

#         vis_mask = np.zeros((*pred_mask.shape, 3), dtype=np.uint8)
#         vis_mask[pred_mask == 1] = [255, 255, 0]
#         vis_mask[pred_mask == 2] = [0, 0, 255]
#         cv2.imwrite(
#             os.path.join(OUTPUT_DIR, f"mask_{img_file}"),
#             cv2.cvtColor(vis_mask, cv2.COLOR_RGB2BGR)
#         )
#         print(f"{img_file}: No Change {no_change:.1f}% | Change {change:.1f}% | Strong Change {strong_change:.1f}%")

#     with open(INFERENCE_RESULTS, 'w') as f:
#         json.dump(inference_results, f, indent=2)
#     print(f"\n추론 완료! {len(inference_results)}개 처리")


# # ============================================================
# # Cell 19: VLM + RAG 전체 추론 → vlm_results.json + KG 저장
# # ============================================================

# vlm_results = {}
# sorted_files = sorted(inference_results.keys())

# for idx, img_file in enumerate(sorted_files, start=1):
#     print(f"[{idx}/{len(sorted_files)}] {img_file}")

#     ratios = inference_results[img_file]
#     no_change     = float(ratios["no_change"])
#     change        = float(ratios["change"])
#     strong_change = float(ratios["strong_change"])
#     total_change  = round(change + strong_change, 2)

#     mask_path = os.path.join(MASK_DIR, f"mask_{img_file}")
#     if not os.path.exists(mask_path):
#         print(f"마스크 없음, 스킵: {mask_path}")
#         continue

#     # RAG 검색
#     total_doc, total_candidates = retrieve_ratio_doc(value=total_change, category="total_change", debug=True)
#     strong_doc, strong_candidates = retrieve_ratio_doc(value=strong_change, category="strong_change", debug=True)

#     rag_context = make_rag_context(
#         img_file=img_file,
#         no_change=no_change, change=change,
#         strong_change=strong_change, total_change=total_change,
#         total_doc=total_doc, strong_doc=strong_doc
#     )

#     # VLM 추론
#     try:
#         response = run_vlm(mask_image_path=mask_path, rag_context=rag_context)
#         vlm_report = clean_vlm_report(response)
#     except Exception as e:
#         print("VLM 추론 실패:", e)
#         response = "VLM failed"
#         vlm_report = "VLM 추론 오류로 RAG 기준으로만 판단하였다."

#     # 최종 결과 생성
#     timestamp = extract_timestamp(img_file)
#     progress = (
#         f"현재 이미지는 No Change {no_change:.2f}%, Change {change:.2f}%, Strong Change {strong_change:.2f}%이다.\n"
#         f"전체 변화율은 {total_change:.2f}%이며, RAG 기준으로 '{total_doc['level']}' 수준이다. {total_doc['text']}\n"
#         f"Strong Change는 {strong_change:.2f}%이며, '{strong_doc['level']}' 수준이다. {strong_doc['text']}"
#     )

#     final_result = {
#         "timestamp": timestamp, "filename": img_file,
#         "total_change_level": total_doc["level"],
#         "strong_change_level": strong_doc["level"],
#         "ratios": {
#             "no_change": round(no_change, 2), "change": round(change, 2),
#             "strong_change": round(strong_change, 2), "total_change": total_change
#         },
#         "progress": progress, "vlm_report": vlm_report, "distribution": vlm_report
#     }

#     vlm_results[img_file] = {
#         "rag_used": {
#             "total_change_doc": total_doc, "strong_change_doc": strong_doc,
#             "total_change_candidates": total_candidates[:3],
#             "strong_change_candidates": strong_candidates[:3],
#             "rag_context": rag_context
#         },
#         "vlm_raw_output": response,
#         "final_output": final_result
#     }

#     # KG 저장
#     try:
#         kg_graph, event_id = append_final_output_to_kg_graph({
#             **final_result,
#             "mask_path": mask_path
#         })
#         print(f"  KG 저장 완료: {event_id}")
#     except Exception as e:
#         print(f"  KG 저장 실패: {e}")

#     print(f"  total_change_level: {final_result['total_change_level']}")
#     print(f"  vlm_report: {vlm_report[:80]}...")

# # vlm_results.json 저장
# with open(VLM_RESULTS_PATH, "w", encoding="utf-8") as f:
#     json.dump(vlm_results, f, ensure_ascii=False, indent=2)

# # KG records 갱신
# refresh_kg_records()

# print(f"\n완료! {len(vlm_results)}개 처리 → {VLM_RESULTS_PATH}")
# print(f"KG records: {len(records)}개")